# E8: Region Transition MDP — Probabilistic Memory (RFC-013)

## Concept

HNSW regions define a **discrete state space**. Episode trajectories define **transitions**.
Rewards define **outcomes**. This is a Markov Decision Process:

| MDP Component | CVX Primitive |
|--------------|---------------|
| State $s$ | `region_assignments(level=1)` |
| Action $a$ | Classified action type (navigate, take, place, clean, heat, cool) |
| Transition $P(s' | s, a)$ | Observed region transitions from episode trajectories |
| Reward $R(s, a)$ | `reward(node_id)` — episode outcome |

## Integration with CVX Retrieval

```
1. CVX retrieves k similar past states (causal_search)
2. MDP scores each action type: P(success | current_region, action_type)
3. Agent sees: strategy + expert actions + "action success rates here: navigate(80%), take(60%)"
```

## Protocol

- Round 1: CVX retrieval only (baseline)
- Rounds 2-5: CVX retrieval + MDP action scoring
- Online learning: MDP updated after each round with new transitions

**Model**: Ollama qwen2.5:7b (zero cost)


In [ ]:
"""
E8: Region Transition MDP — Probabilistic memory over HNSW regions

Proof of concept for RFC-013 Part A: learn P(success | region, action_type)
from episode trajectories, use it to rank retrieved actions.

Protocol:
- Round 1: Play with CVX retrieval only (baseline = E7b Round 1)
- Round 2-N: Play with CVX retrieval + MDP scoring
- Online learning: update transition model after each round

Model: Ollama qwen2.5:7b (zero cost)
"""
import os, json, time, re
from collections import Counter, defaultdict
import numpy as np
from sentence_transformers import SentenceTransformer
from openai import OpenAI

OLLAMA_URL = 'http://hpc.glaciar.lab:11434/v1'
client = OpenAI(base_url=OLLAMA_URL, api_key='ollama')
LLM_MODEL = 'qwen2.5-coder:7b-instruct'

MAX_STEPS = 30
N_EVAL = 30
N_ROUNDS = 5
TOP_K = 3
REGION_LEVEL = 1

embedder = SentenceTransformer('all-MiniLM-L6-v2')
print(f'LLM: {LLM_MODEL} (Ollama)')

import chronos_vector as cvx

# === ACTION TYPE CLASSIFICATION ===
def classify_action(action_text):
    """Classify action into abstract type."""
    a = action_text.lower()
    if a.startswith('go to'): return 'navigate'
    if a.startswith('take') or a.startswith('pick'): return 'take'
    if a.startswith('put'): return 'place'
    if a.startswith('open'): return 'open'
    if a.startswith('close'): return 'close'
    if a.startswith('clean'): return 'clean'
    if a.startswith('heat') or a.startswith('cook'): return 'heat'
    if a.startswith('cool'): return 'cool'
    if a.startswith('use'): return 'use'
    if a.startswith('examine') or a.startswith('look'): return 'examine'
    return 'other'

# === TASK TYPE ===
TASK_TYPES = {
    'heat': 'heat_then_place', 'cool': 'cool_then_place',
    'clean': 'clean_then_place', 'examine': 'examine_in_light',
    'look at': 'examine_in_light', 'find two': 'pick_two',
    'put two': 'pick_two',
}
def detect_task_type(task):
    for kw, tt in TASK_TYPES.items():
        if kw in task.lower(): return tt
    return 'pick_and_place'

STRATEGIES = {
    'pick_and_place': 'Find the object, take it, go to target, put it.',
    'heat_then_place': 'Find object, take, go to microwave, heat, take, go to target, put.',
    'cool_then_place': 'Find object, take, go to fridge, cool, take, go to target, put.',
    'clean_then_place': 'Find object, take, go to sinkbasin, clean, go to target, put.',
    'examine_in_light': 'Find object, take, find lamp, use lamp.',
    'pick_two': 'Find first, take, put at target. Find second, take, put at target.',
}

# === REGION MDP ===
class RegionMDP:
    """MDP over HNSW regions learned from episode trajectories."""

    def __init__(self):
        self.transitions = defaultdict(Counter)  # (region, action_type) → {next_region: count}
        self.rewards = defaultdict(list)          # (region, action_type) → [rewards]
        self.region_success = defaultdict(list)   # region → [episode_rewards]

    def learn_episode(self, index, traj_nodes, actions, reward):
        """Learn from one episode's trajectory."""
        for i in range(len(traj_nodes) - 1):
            vec_i = index.trajectory(traj_nodes[i])[0][1] if False else None
            # Use node vectors to assign regions
            # Simplified: use the trajectory timestamps to get vectors
            pass

    def learn_from_data(self, region_sequence, action_sequence, episode_reward):
        """Learn from a sequence of (region, action) pairs."""
        for i in range(len(region_sequence) - 1):
            s = region_sequence[i]
            a = action_sequence[i]
            s_next = region_sequence[i + 1]

            self.transitions[(s, a)][s_next] += 1
            self.rewards[(s, a)].append(episode_reward)

        # Track per-region success
        for s in set(region_sequence):
            self.region_success[s].append(episode_reward)

    def action_success_rate(self, region, action_type):
        """P(success | region, action_type)"""
        rewards = self.rewards.get((region, action_type), [])
        if not rewards:
            return 0.5  # uninformative prior
        return np.mean(rewards)

    def best_actions(self, region):
        """Rank action types by success rate in this region."""
        scores = {}
        for (s, a), rewards in self.rewards.items():
            if s == region and len(rewards) >= 2:
                scores[a] = np.mean(rewards)
        return sorted(scores.items(), key=lambda x: -x[1])

    def region_quality(self, region):
        """Overall success rate of episodes passing through this region."""
        r = self.region_success.get(region, [])
        return np.mean(r) if r else 0.5

    def stats(self):
        n_transitions = sum(sum(c.values()) for c in self.transitions.values())
        n_states = len(set(s for (s, a) in self.transitions.keys()))
        n_actions = len(set(a for (s, a) in self.transitions.keys()))
        return f'{n_transitions} transitions, {n_states} states, {n_actions} action types'

# === BUILD INDEX ===
with open('data/episodic/alfworld_metadata.json') as f:
    meta = json.load(f)

print('Building index...')
index = cvx.TemporalIndex(m=16, ef_construction=100)
action_lookup, task_lookup = {}, {}

for ep_id_str, ep in meta.items():
    ep_id = int(ep_id_str)
    task = ep.get('task', '')
    task_lookup[ep_id] = task
    for si, step in enumerate(ep['steps']):
        text = f"{task} | {step.get('action','')} | {step.get('observation','')}"
        vec = embedder.encode(text).tolist()
        ts = ep_id * 10000 + si
        index.insert(ep_id, ts, vec, reward=1.0)
        action_lookup[ts] = step.get('action', '')

print(f'Index: {len(index)} points')

# === LEARN INITIAL MDP FROM EXPERT TRAJECTORIES ===
mdp = RegionMDP()
assignments_cache = index.region_assignments(REGION_LEVEL)

# Build node → region mapping
node_to_region = {}
for hub_id, members in assignments_cache.items():
    for eid, ts in members:
        node_to_region[ts] = hub_id  # ts is unique per node in our encoding

# Learn from expert episodes
for ep_id_str, ep in meta.items():
    ep_id = int(ep_id_str)
    steps = ep.get('steps', [])
    regions = []
    actions = []
    for si, step in enumerate(steps):
        ts = ep_id * 10000 + si
        r = node_to_region.get(ts, -1)
        regions.append(r)
        actions.append(classify_action(step.get('action', '')))
    if len(regions) >= 2:
        mdp.learn_from_data(regions, actions, 1.0)  # experts are all successful

print(f'MDP: {mdp.stats()}')

# === CONTEXT FORMATTING WITH MDP SCORING ===
def format_context_with_mdp(results, task_type, current_region, mdp):
    if not results: return ''
    strategy = STRATEGIES.get(task_type, '')
    parts = []
    if strategy:
        parts.append(f'Strategy: {strategy}')

    # MDP insights
    best = mdp.best_actions(current_region)
    if best:
        action_hints = ', '.join(f'{a}({s:.0%})' for a, s in best[:3])
        parts.append(f'Action success rates here: {action_hints}')

    parts.append('Expert sequences:')
    for i, r in enumerate(results[:TOP_K]):
        actions = []
        for nid, ts, vec in r.get('successors', [])[:5]:
            a = action_lookup.get(ts, '')
            if a: actions.append(a)
        if actions:
            parts.append(f'  [{i+1}] {" -> ".join(actions)}')
    return '\n'.join(parts) + '\n'

# === AGENT ===
def llm_call(system, user):
    try:
        resp = client.chat.completions.create(
            model=LLM_MODEL,
            messages=[{'role':'system','content':system},{'role':'user','content':user}],
            temperature=0.0, max_tokens=100,
        )
        return resp.choices[0].message.content.strip()
    except: return ''

def agent_act(observation, admissible, task, history, context=''):
    system = 'You are a household agent. Choose ONE action from the list. Output ONLY the action.'
    user = f'Task: {task}\n\n'
    if context: user += context + '\n'
    if history:
        user += 'Recent:\n' + '\n'.join(f'  > {h["a"]} -> {h["o"][:60]}' for h in history[-5:]) + '\n\n'
    user += f'Obs: {observation}\nActions:\n' + '\n'.join(f'  - {a}' for a in admissible) + '\nChoose:'
    chosen = llm_call(system, user).lower().strip()
    for a in admissible:
        if a.lower() == chosen: return a
    for a in admissible:
        if a.lower() in chosen or chosen in a.lower(): return a
    return admissible[0]

def get_node_region(vec):
    """Assign a vector to a region (approximate via nearest search)."""
    results = index.search(vec, k=1)
    if results:
        eid, ts, score = results[0]
        return node_to_region.get(ts, -1)
    return -1

def run_game(env, index, mdp, use_mdp=False):
    obs, info = env.reset()
    observation = obs[0]
    task = observation.split('Your task is to:')[-1].strip().split('\n')[0] if 'Your task is to:' in observation else ''
    task_type = detect_task_type(task)
    history = []
    step_regions, step_actions = [], []

    for step in range(MAX_STEPS):
        admissible = info['admissible_commands'][0]
        ctx = ''
        try:
            qv = embedder.encode(f'{task} | {observation}').tolist()
            results = index.causal_search(vector=qv, k=TOP_K, temporal_context=5)
            current_region = get_node_region(qv)
            if use_mdp:
                ctx = format_context_with_mdp(results, task_type, current_region, mdp)
            else:
                # Compact format without MDP
                parts = [f'Strategy: {STRATEGIES.get(task_type, "")}', 'Expert sequences:']
                for i, r in enumerate(results[:TOP_K]):
                    actions = [action_lookup.get(ts, '') for nid, ts, vec in r.get('successors', [])[:5] if action_lookup.get(ts)]
                    if actions: parts.append(f'  [{i+1}] {" -> ".join(actions)}')
                ctx = '\n'.join(parts) + '\n'
        except:
            current_region = -1

        action = agent_act(observation, admissible, task, history, ctx)
        obs, rewards, dones, info = env.step([action])
        observation = obs[0]
        history.append({'a': action, 'o': observation[:150]})
        step_regions.append(current_region)
        step_actions.append(classify_action(action))
        if dones[0]: break

    won = info['won'][0]
    return {
        'task': task, 'task_type': task_type, 'won': won,
        'steps': len(history), 'history': history,
        'regions': step_regions, 'actions': step_actions,
    }

# === ALFWORLD ===
from alfworld.agents.environment.alfred_tw_env import AlfredTWEnv
config = {
    'dataset': {
        'data_path': os.path.expanduser('~/.cache/alfworld/json_2.1.1/train'),
        'eval_id_data_path': os.path.expanduser('~/.cache/alfworld/json_2.1.1/valid_seen'),
        'eval_ood_data_path': os.path.expanduser('~/.cache/alfworld/json_2.1.1/valid_unseen'),
        'num_train_games': 0, 'num_eval_games': 0,
    },
    'env': {'goal_desc_human_anns_prob': 0.0, 'task_types': [1,2,3,4,5,6],
            'domain_randomization': False, 'expert_type': 'handcoded'},
    'general': {'training_method': 'dqn', 'random_seed': 42},
    'rl': {'training': {'max_nb_steps_per_episode': MAX_STEPS}},
    'dagger': {'training': {'max_nb_steps_per_episode': MAX_STEPS}},
    'logic': {'domain': os.path.expanduser('~/.cache/alfworld/logic/alfred.pddl'),
              'grammar': os.path.expanduser('~/.cache/alfworld/logic/alfred.twl2')},
}
env_wrapper = AlfredTWEnv(config, train_eval='eval_out_of_distribution')
env = env_wrapper.init_env(batch_size=1)
print(f'ALFWorld: {len(env_wrapper.game_files)} games\n')

# === MAIN LOOP ===
all_results = {}
next_ep_id = 1000

for rnd in range(N_ROUNDS):
    use_mdp = rnd > 0  # Round 1 = baseline (no MDP), Round 2+ = with MDP
    label = f'Round{rnd+1}{"_MDP" if use_mdp else "_base"}'

    print(f'\n{"#"*60}')
    print(f'{label} — Index: {len(index)}, MDP: {mdp.stats()}')
    print(f'{"#"*60}')

    results = []
    wins = 0
    for g in range(N_EVAL):
        r = run_game(env, index, mdp, use_mdp=use_mdp)
        results.append(r)
        if r['won']: wins += 1
        print(f'  {g+1}/{N_EVAL}: {"WIN" if r["won"] else "fail"} ({r["steps"]}s) {r["task_type"][:15]} [{wins}/{g+1}={wins/(g+1)*100:.0f}%]', flush=True)

    rate = wins / N_EVAL * 100
    print(f'\n  >>> {label}: {wins}/{N_EVAL} = {rate:.1f}%')
    all_results[label] = [{'won':r['won'],'steps':r['steps'],'task':r['task'],'task_type':r['task_type']} for r in results]

    # Online learning: add experience + update MDP
    for r in results:
        rw = 1.0 if r['won'] else 0.0
        for si, step in enumerate(r['history']):
            vec = embedder.encode(f"{r['task']} | {step['a']} | {step['o']}").tolist()
            ts = next_ep_id * 10000 + si
            index.insert(next_ep_id, ts, vec, reward=rw)
            action_lookup[ts] = step['a']
            node_to_region[ts] = get_node_region(vec)
        task_lookup[next_ep_id] = r['task']

        # Update MDP with this episode
        if len(r['regions']) >= 2:
            mdp.learn_from_data(r['regions'], r['actions'], rw)

        next_ep_id += 1

    # Refresh region assignments (regions may shift with new data)
    new_assignments = index.region_assignments(REGION_LEVEL)
    for hub_id, members in new_assignments.items():
        for eid, ts in members:
            node_to_region[ts] = hub_id

    print(f'  MDP updated: {mdp.stats()}')

    with open('data/episodic/e8_mdp_results.json', 'w') as f:
        json.dump(all_results, f, indent=2)

# === SUMMARY ===
print(f'\n{"="*60}')
print('REGION MDP RESULTS')
print(f'{"="*60}')
for label, res in all_results.items():
    w = sum(1 for r in res if r['won'])
    print(f'  {label}: {w}/{len(res)} = {w/len(res)*100:.1f}%')

print(f'\nMDP action success rates (top regions):')
region_counts = Counter(node_to_region.values())
for region, count in region_counts.most_common(5):
    best = mdp.best_actions(region)
    if best:
        print(f'  Region {region} ({count} nodes): {", ".join(f"{a}={s:.0%}" for a,s in best[:3])}')

print('\nSaved to data/episodic/e8_mdp_results.json')
